# Métriques d'évaluation pour la segmentation

Master Data & IA : évaluation en vision par ordinateur.

Ce notebook présente les métriques de **segmentation sémantique et d'instance** sur des masques synthétiques : matrice de confusion, précision, rappel, F1/Dice, IoU, courbes ROC/PR, AP d'instance, et un contre-exemple avec MSE/RMSE/R².

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score, mean_squared_error, r2_score)

rng = np.random.default_rng(0)
NUM_CLASSES = 3  # 0 = fond, 1 = route, 2 = voiture

## 1. Données synthétiques
Vérité terrain à 3 classes (fond, route, voiture) et prédiction décalée avec du bruit.

In [ ]:
def make_data(h=128, w=128):
    gt = np.zeros((h, w), dtype=int)
    gt[40:100, 10:118] = 1          # route
    gt[55:80, 30:70] = 2            # voiture
    # "prédiction" = vérité terrain bruitée + décalage
    pred = np.roll(gt, shift=(3, 4), axis=(0, 1))
    noise = rng.random((h, w)) < 0.05
    pred[noise] = rng.integers(0, NUM_CLASSES, noise.sum())
    return gt, pred

## 2. Matrice de confusion et métriques par classe
Précision = TP/(TP+FP), rappel = TP/(TP+FN), F1/Dice = 2TP/(2TP+FP+FN), IoU = TP/(TP+FP+FN).

In [ ]:
def confusion_matrix(gt, pred, n=NUM_CLASSES):
    idx = n * gt.flatten() + pred.flatten()
    return np.bincount(idx, minlength=n * n).reshape(n, n)


def per_class_metrics(cm):
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    eps = 1e-9
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)   # = Dice
    iou = tp / (tp + fp + fn + eps)                            # = Jaccard
    pixel_acc = tp.sum() / cm.sum()
    return dict(precision=precision, recall=recall, f1_dice=f1,
                iou=iou, miou=iou.mean(), pixel_acc=pixel_acc)

## 3. Segmentation d'instance
IoU entre masques, appariement par score décroissant, AP à un seuil d'IoU donné.

In [ ]:
def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union else 0.0


def average_precision(gt_masks, pred_masks, scores, iou_thr=0.5):
    """AP à un seuil d'IoU (matching glouton par score décroissant)."""
    order = np.argsort(scores)[::-1]
    matched, tp = set(), []
    for i in order:
        best, best_j = 0, -1
        for j, g in enumerate(gt_masks):
            if j in matched:
                continue
            iou = mask_iou(pred_masks[i], g)
            if iou > best:
                best, best_j = iou, j
        if best >= iou_thr:
            matched.add(best_j)
            tp.append(1)
        else:
            tp.append(0)
    tp = np.array(tp)
    cum_tp, cum_fp = np.cumsum(tp), np.cumsum(1 - tp)
    rec = cum_tp / len(gt_masks)
    prec = cum_tp / (cum_tp + cum_fp)
    # intégration de la courbe précision-rappel
    mrec = np.concatenate([[0], rec, [1]])
    mpre = np.concatenate([[0], prec, [0]])
    for k in range(len(mpre) - 2, -1, -1):
        mpre[k] = max(mpre[k], mpre[k + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


def instance_demo():
    def box(y0, y1, x0, x1):
        m = np.zeros((100, 100), bool)
        m[y0:y1, x0:x1] = True
        return m
    gts = [box(10, 40, 10, 40), box(50, 90, 50, 90)]
    preds = [box(12, 42, 12, 42), box(60, 90, 50, 95), box(0, 10, 0, 10)]
    scores = [0.9, 0.7, 0.4]
    return {t: average_precision(gts, preds, scores, t) for t in (0.5, 0.75)}

## 4. Courbes ROC et précision-rappel
Probabilités simulées pour la classe voiture. La PR est plus informative quand le fond domine.

In [ ]:
def roc_pr_demo(gt):
    y_true = (gt == 2).flatten().astype(int)
    # probabilités simulées : vérité + bruit gaussien
    y_score = np.clip(y_true * 0.6 + rng.normal(0.2, 0.2, y_true.size), 0, 1)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.3f}")
    ax[0].plot([0, 1], [0, 1], "--", c="gray")
    ax[0].set(title="Courbe ROC (classe voiture)", xlabel="FPR", ylabel="TPR")
    ax[0].legend()
    ax[1].plot(rec, prec, label=f"AP = {average_precision_score(y_true, y_score):.3f}")
    ax[1].set(title="Courbe Précision-Rappel", xlabel="Rappel", ylabel="Précision")
    ax[1].legend()
    plt.tight_layout()
    plt.savefig("roc_pr.png", dpi=150)
    plt.close()
    return y_true, y_score


def plot_confusion(cm, names=("fond", "route", "voiture")):
    cmn = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cmn, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES), names)
    ax.set_yticks(range(NUM_CLASSES), names)
    ax.set(xlabel="Prédit", ylabel="Réel", title="Matrice de confusion (normalisée)")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.close()

## 5. MSE / RMSE / R²
Métriques de régression : calculées ici seulement pour montrer qu'elles ne conviennent pas à la segmentation.

In [ ]:
def regression_metrics_demo(y_true, y_score):
    mse = mean_squared_error(y_true, y_score)
    return dict(MSE=mse, RMSE=np.sqrt(mse), R2=r2_score(y_true, y_score))

## Exécution de la démo

In [ ]:
gt, pred = make_data()
cm = confusion_matrix(gt, pred)
m = per_class_metrics(cm)
print("Matrice de confusion :\n", cm)
for k, v in m.items():
    print(f"{k:>10}: {np.round(v, 4)}")
plot_confusion(cm)

In [ ]:
print("AP instance :", instance_demo())

In [ ]:
y_true, y_score = roc_pr_demo(gt)
print("Régression :", {k: round(float(v), 4) for k, v in regression_metrics_demo(y_true, y_score).items()})

In [ ]:
from IPython.display import Image, display
display(Image("confusion_matrix.png")); display(Image("roc_pr.png"))